# MultiModal

In [21]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [22]:
import base64
from typing import Literal
from pydantic import BaseModel, Field


# Image Encoding

In [23]:
def encode_image(image_path):
    """Convert image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')
    
    
#Prepare the image
image_path = "./img/junction_sample.png"
image_base64 = encode_image(image_path)

# Prompts For Model

In [24]:
system_navigation_prompt = SystemMessage(
    content="""You are an AGV Navigation Controller for a smart warehouse. 
    Your task is to analyze warehouse floor images and guide an AGV along painted yellow floor lines.

    ### CRITICAL INSTRUCTIONS:
    1. IDENTIFY: Locate the AGV's current position relative to the yellow line network.
    2. DETECT JUNCTIONS: Identify if the AGV is at a 'T-junction', 'Cross-junction', or 'Corner'.
    3. TARGET: The user will provide a destination. You must determine the correct path.
    4. OUTPUT FORMAT: Respond ONLY with a single-word direction (LEFT, RIGHT, STRAIGHT, or STOP) followed by a brief 1-sentence reason.

    ### RULES:
    - If no yellow line is visible, output: "STOP - Line lost."
    - If the AGV is not yet at a junction, output: "STRAIGHT - Following line."
    - Assume the bottom of the image is the AGV's current heading."""
)

In [34]:

# Construct the Multimodal Message
# Note: Gemma 3 via Ollama expects the standard 'image_url' format
input = HumanMessage(
    content=[
        {"type": "text", "text": "Which direction should i move to reach assembly line A"},
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{image_base64}"},
        },
    ]
)

messages = [system_navigation_prompt,input]

# Calling The Model

In [35]:
model = init_chat_model(
    model="gemma3:4b",
    model_provider="ollama",
    temperature = 0
    )

## Ouput Format For The Model

In [36]:
class JunctionAssistor(BaseModel):
    direction : Literal['right','left','straight'] = Field(description='The direction the agv should move')
    reason : str = Field(description='Reason for the direction given')

In [37]:
#  Invoke the model
junction_assistance = model.with_structured_output(JunctionAssistor)

response = junction_assistance.invoke(messages)
print(response)
print(response.direction)

direction='left' reason='The AGV is at a T-junction and needs to turn left to follow the yellow line towards Assembly Line A.'
left
